# Conservative classifier self-training from the frozen DBSCAN seed

Keep the current DBSCAN + logistic regression model as the seed. Generate additional partial labels only for **unlabeled training reason slots that the seed dataset leaves unknown**. Never replace known labels or existing DBSCAN pseudo-labels. Validation/test trials never enter self-training.

Predeclared grid: positive confidence >=**0.90 or 0.95**, negative confidence <=**0.10 or 0.05** respectively; added per-example weight cap **0.05 or 0.10**; evaluate **one and two rounds** (8 variants). Each round refreshes self-generated labels from the original seed's unknown slots using the preceding round's classifier. Both pseudo classes must exist for a reason. Their added raw weight fractions preserve the known class ratio and are capped at **0.25** of each known class's weight, with total seed+self influence <=0.50. Existing DBSCAN targets and raw weights stay intact.

Classifier scaler, 150 embeddings, C=1000, normalized fitting-weight total 157 per reason, final cutoff 0.5, and highest-probability fallback stay fixed. Model confidence is an acceptance heuristic, not a guarantee of correctness. No final train+validation refit, Kaggle submission, commit, or tag.

In [1]:
from pathlib import Path
import hashlib
import json
import platform
import warnings
import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import NearestNeighbors, KNeighborsClassifier
from threadpoolctl import threadpool_limits

SEED = 42
REASONS = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
FEATURES = [f"emb_{i}" for i in range(150)]
LABEL_RULE = {'minimum_support': 3.0, 'prior_strength': 2.0, 'positive_threshold': 0.6, 'negative_threshold': 0.1, 'maximum_generated_weight': 0.25, 'maximum_class_fraction': 0.5}
CONFIDENCES=[0.90,0.95]
SELF_WEIGHT_CAPS=[0.05,0.10]
ROUNDS=2
MAXIMUM_ADDED_CLASS_FRACTION=0.25
from sklearn.base import clone
cwd = Path.cwd()
DATA = next(q for q in [cwd, cwd/"part1", cwd/"AML-Midterm"/"part1"]
            if (q/"part1_train.csv").is_file() or (q/"data"/"part1_train.csv").is_file()).resolve()
INPUT_DIR = DATA if (DATA/"part1_train.csv").is_file() else DATA/"data"
OUT = DATA/"experiments"/"self_training"
for folder in ["datasets", "label_audits", "clusters", "clustering_models", "classifier_models", "validation_predictions", "submissions", "labeling_models"]:
    (OUT/folder).mkdir(parents=True, exist_ok=True)
train = pd.read_csv(INPUT_DIR/"part1_train.csv")
valid = pd.read_csv(INPUT_DIR/"part1_val.csv")
X = train[FEATURES].to_numpy()
X_valid = valid[FEATURES].to_numpy()
truth = valid[REASONS].to_numpy(dtype=int)
original_y = train[REASONS].to_numpy(dtype=int)
known = train[REASONS].isin([0,1]).all(axis=1).to_numpy()
assert known.sum()==157 and len(train)==1171 and len(valid)==78
assert train.ID.is_unique and valid.ID.is_unique and set(train.ID).isdisjoint(valid.ID)
assert np.isfinite(X).all() and np.isfinite(X_valid).all()
assert np.isin(truth,[0,1]).all()
baseline = joblib.load(DATA/"experiments/kmeans/models/baseline_C1000.joblib")
legacy_packet = joblib.load(DATA/"experiments/kmeans/models/k02.joblib")
classifier_scaler = baseline.named_steps["scaler"]
cluster_scaler = legacy_packet["kmeans_scaler"]
assert classifier_scaler.n_samples_seen_==157 and cluster_scaler.n_samples_seen_==1171
XS = classifier_scaler.transform(X)
VS = classifier_scaler.transform(X_valid)
XC = cluster_scaler.transform(X)
protected_paths = [INPUT_DIR/"part1_train.csv", INPUT_DIR/"part1_val.csv"] + sorted(DATA.glob("*submission.csv")) + [
    DATA/"experiments/kmeans/models/k02.joblib", DATA/"experiments/kmeans/datasets/k02_train.csv"
]
protected_hashes = {str(q.relative_to(DATA)):hashlib.sha256(q.read_bytes()).hexdigest() for q in protected_paths}
print("Using 150 features; 157 known-label and 1,014 unlabeled training trials; 78 validation trials.")

Using 150 features; 157 known-label and 1,014 unlabeled training trials; 78 validation trials.


In [2]:
def label_predictions(probabilities):
    probabilities = np.asarray(probabilities)
    assert probabilities.shape[1]==5 and np.isfinite(probabilities).all()
    assert ((probabilities>=0)&(probabilities<=1)).all()
    prediction=(probabilities>=0.5).astype(int)
    empty=prediction.sum(axis=1)==0
    ids=np.flatnonzero(empty)
    prediction[ids, probabilities[ids].argmax(axis=1)]=1
    return prediction,empty


def score_probabilities(probabilities):
    predicted,empty=label_predictions(probabilities)
    return {"sample_f1":f1_score(truth,predicted,average="samples",zero_division=0),
            "macro_f1":f1_score(truth,predicted,average="macro",zero_division=0),
            "fallback_trials":int(empty.sum()),
            **dict(zip([f"f1_{r}" for r in REASONS],f1_score(truth,predicted,average=None,zero_division=0)))},predicted,empty


def save_predictions(name, probabilities):
    metrics,predicted,empty=score_probabilities(probabilities)
    output=valid[["ID"]+REASONS].rename(columns={r:f"actual_{r}" for r in REASONS}).copy()
    for j,r in enumerate(REASONS):
        output[f"probability_{r}"]=probabilities[:,j]
        output[f"predicted_{r}"]=predicted[:,j]
    output["used_fallback"]=empty
    output.to_csv(OUT/"validation_predictions"/f"{name}.csv",index=False)
    return metrics


DATASETS={}
LABEL_COUNTS=[]
FIT_CACHE={}


def save_dataset(name, targets, weights, probabilities=None, component=None, support=None):
    assert targets.shape==weights.shape==(len(train),5)
    np.testing.assert_array_equal(targets[known],original_y[known])
    np.testing.assert_array_equal(weights[known],np.ones((known.sum(),5)))
    assert np.isin(targets,[-100,0,1]).all() and (weights>=0).all()
    assert np.array_equal(weights>0,targets!=-100)
    frame=train[["ID"]+FEATURES].copy()
    audit=train[["ID"]].copy()
    for j,r in enumerate(REASONS):
        source=np.where(known,"original",np.where(weights[:,j]>0,"generated","unknown"))
        frame[r]=targets[:,j]
        frame[f"weight_{r}"]=weights[:,j]
        audit[r]=targets[:,j]
        audit[f"original_{r}"]=original_y[:,j]
        audit[f"weight_{r}"]=weights[:,j]
        audit[f"label_source_{r}"]=source
        if probabilities is not None:audit[f"estimated_probability_{r}"]=probabilities[:,j]
        pos=(~known)&(targets[:,j]==1)
        neg=(~known)&(targets[:,j]==0)
        LABEL_COUNTS.append({"dataset":name,"reason":r,
            "known_positives":int(original_y[known,j].sum()),
            "known_negatives":int((original_y[known,j]==0).sum()),
            "generated_positives":int(pos.sum()),"generated_negatives":int(neg.sum()),
            "generated_positive_weight":float(weights[pos,j].sum()),
            "generated_negative_weight":float(weights[neg,j].sum())})
    if component is not None:audit["component"]=component
    if support is not None:audit["effective_labeled_support"]=support
    frame["included_for_any_reason"]=(weights>0).any(axis=1)
    frame.to_csv(OUT/"datasets"/f"{name}.csv",index=False)
    audit.to_csv(OUT/"label_audits"/f"{name}.csv",index=False)
    DATASETS[name]={"targets":targets.copy(),"weights":weights.copy(),
                    "generated_trials":int(((weights>0).any(axis=1)&~known).sum())}
    return DATASETS[name]


def partial_labels(responsibilities):
    assert responsibilities.shape[0]==len(train)
    np.testing.assert_allclose(responsibilities.sum(axis=1),1,atol=1e-9)
    component_support=responsibilities[known].sum(axis=0)
    positives=responsibilities[known].T@original_y[known]
    prior=original_y[known].mean(axis=0)
    rate=(positives+LABEL_RULE["prior_strength"]*prior)/(component_support[:,None]+LABEL_RULE["prior_strength"])
    probabilities=responsibilities@rate
    support=responsibilities@component_support
    targets=np.full(original_y.shape,-100,dtype=int)
    weights=np.zeros(original_y.shape,dtype=float)
    targets[known]=original_y[known]
    weights[known]=1.0
    eligible=(~known)&(support>=LABEL_RULE["minimum_support"])
    for j in range(5):
        positive=eligible&(probabilities[:,j]>=LABEL_RULE["positive_threshold"])
        negative=eligible&(probabilities[:,j]<=LABEL_RULE["negative_threshold"])
        npos,nneg=int(positive.sum()),int(negative.sum())
        kp=int(original_y[known,j].sum());kn=int(known.sum()-kp)
        if not npos or not nneg:continue
        fraction=min(LABEL_RULE["maximum_class_fraction"],
                     LABEL_RULE["maximum_generated_weight"]*npos/kp,
                     LABEL_RULE["maximum_generated_weight"]*nneg/kn)
        targets[positive,j]=1;targets[negative,j]=0
        weights[positive,j]=fraction*kp/npos
        weights[negative,j]=fraction*kn/nneg
        assert weights[~known,j].max()<=LABEL_RULE["maximum_generated_weight"]+1e-12
        np.testing.assert_allclose(weights[positive,j].sum()/kp,weights[negative,j].sum()/kn,atol=1e-12)
    components=pd.DataFrame({"component":np.arange(len(component_support)),"effective_labeled_support":component_support,
                             "total_membership_mass":responsibilities.sum(axis=0)})
    for j,r in enumerate(REASONS):
        components[f"weighted_known_positive_{r}"]=positives[:,j]
        components[f"estimated_probability_{r}"]=rate[:,j]
    return targets,weights,probabilities,support,components


def fit_reason_models(dataset_name, kind="logistic", parameter=1000.0):
    dataset=DATASETS[dataset_name]
    estimators=[]
    for j in range(5):
        y=dataset["targets"][:,j];w=dataset["weights"][:,j]
        included=w>0
        assert np.isin(y[included],[0,1]).all() and included[known].all()
        key=(kind,float(parameter),j,hashlib.sha256(y.tobytes()+w.tobytes()).hexdigest())
        if key not in FIT_CACHE:
            if kind=="logistic":
                model=LogisticRegression(C=float(parameter),solver="lbfgs",max_iter=5000,tol=1e-4,random_state=SEED)
                normalized=w[included]*known.sum()/w[included].sum()
                with warnings.catch_warnings():
                    warnings.simplefilter("error",ConvergenceWarning)
                    model.fit(XS[included],y[included],sample_weight=normalized)
            else:
                model={"neighbors":NearestNeighbors(n_neighbors=min(int(parameter),int(included.sum())),algorithm="brute").fit(XS[included]),
                       "target":y[included].copy(),"sample_weight":w[included].copy()}
            FIT_CACHE[key]=model
        estimators.append(FIT_CACHE[key])
    return {"kind":kind,"parameter":float(parameter),"scaler":classifier_scaler,"estimators":estimators,
            "features":FEATURES,"reasons":REASONS,"dataset":dataset_name,
            "logistic_weight_normalization_total":int(known.sum())}


def predict_reason_models(model, raw_features):
    scaled=model["scaler"].transform(np.asarray(raw_features))
    output=[]
    for estimator in model["estimators"]:
        if model["kind"] in ["logistic", "rbf_svm", "gaussian_nb"]:
            positive=list(estimator.classes_).index(1)
            probability=estimator.predict_proba(scaled)[:,positive]
        else:
            distance,indices=estimator["neighbors"].kneighbors(scaled)
            zero=distance==0
            exact=zero.any(axis=1)
            distance_weight=np.divide(1.,distance,out=np.zeros_like(distance),where=distance!=0)
            distance_weight[exact]=zero[exact].astype(float)
            vote=distance_weight*estimator["sample_weight"][indices]
            probability=(vote*estimator["target"][indices]).sum(axis=1)/vote.sum(axis=1)
        output.append(probability)
    probabilities=np.column_stack(output)
    assert np.isfinite(probabilities).all()
    return np.clip(probabilities,0,1)


known_y=np.full_like(original_y,-100);known_y[known]=original_y[known]
known_w=np.zeros(original_y.shape);known_w[known]=1.0
save_dataset("known_only",known_y,known_w)
with threadpool_limits(limits=1):
    reproduced=fit_reason_models("known_only")
np.testing.assert_allclose(predict_reason_models(reproduced,X_valid),baseline.predict_proba(X_valid),rtol=1e-8,atol=1e-10)
reference_rows=[]
for name,model,kaggle in [("submitted_supervised",baseline,0.87072),("submitted_k02",legacy_packet["classifier"],0.87393)]:
    metrics=save_predictions(name,model.predict_proba(X_valid))
    reference_rows.append({"experiment":name,"dataset":"frozen reference","classifier":"logistic","parameter":1000.0,
                           "generated_trials":0 if name=="submitted_supervised" else 388,**metrics,"kaggle_score":kaggle})
print("Confirmed that the new per-reason trainer reproduces the supervised reference.")

Confirmed that the new per-reason trainer reproduces the supervised reference.


In [3]:
SOURCE=DATA/"experiments/additional_methods"
source_record=json.loads((SOURCE/"submission_manifest.json").read_text())
seed_model=joblib.load(SOURCE/source_record["model_file"])
seed_frame=pd.read_csv(SOURCE/"datasets/dbscan_m5_q25.csv",float_precision="round_trip")
assert seed_frame.ID.tolist()==train.ID.tolist()
seed_y=seed_frame[REASONS].to_numpy(dtype=int)
seed_w=seed_frame[[f"weight_{r}" for r in REASONS]].to_numpy()
save_dataset("dbscan_seed",seed_y,seed_w)
with threadpool_limits(limits=1):reproduced_seed=fit_reason_models("dbscan_seed")
seed_probability=predict_reason_models(seed_model,X_valid)
np.testing.assert_allclose(predict_reason_models(reproduced_seed,X_valid),seed_probability,rtol=1e-8,atol=1e-10)
seed_metrics=save_predictions("dbscan_seed",seed_probability)
np.testing.assert_allclose(seed_metrics["sample_f1"],0.9102564102564102,atol=1e-14)
seed_prediction,_=label_predictions(seed_probability)
for p in [SOURCE/source_record["model_file"],SOURCE/"datasets/dbscan_m5_q25.csv",SOURCE/source_record["submission_file"],SOURCE/"labeling_models/dbscan_m5_q25.joblib"]:
    protected_hashes[str(p.relative_to(DATA))]=hashlib.sha256(p.read_bytes()).hexdigest()
ROWS=[];ACCEPTANCE=[];MODELS={"dbscan_seed":seed_model}


def build_self_targets(teacher_probability,confidence,cap):
    targets=seed_y.copy();weights=seed_w.copy()
    unknown=(~known)[:,None]&(seed_w==0)
    for j,r in enumerate(REASONS):
        positive=unknown[:,j]&(teacher_probability[:,j]>=confidence)
        negative=unknown[:,j]&(teacher_probability[:,j]<=1-confidence)
        npos,nneg=int(positive.sum()),int(negative.sum())
        kp=int(original_y[known,j].sum());kn=int(known.sum()-kp)
        seeded_pos=seed_w[(~known)&(seed_y[:,j]==1),j].sum()/kp
        seeded_neg=seed_w[(~known)&(seed_y[:,j]==0),j].sum()/kn
        np.testing.assert_allclose(seeded_pos,seeded_neg,atol=1e-12)
        fraction=0.0
        if npos and nneg:
            fraction=min(MAXIMUM_ADDED_CLASS_FRACTION,max(0.0,0.5-max(seeded_pos,seeded_neg)),cap*npos/kp,cap*nneg/kn)
            if fraction>0:
                targets[positive,j]=1;targets[negative,j]=0
                weights[positive,j]=fraction*kp/npos;weights[negative,j]=fraction*kn/nneg
        added=(weights[:,j]>0)&(seed_w[:,j]==0)
        assert weights[added,j].max(initial=0)<=cap+1e-12
        pos_added=added&(targets[:,j]==1);neg_added=added&(targets[:,j]==0)
        np.testing.assert_allclose(weights[pos_added,j].sum()/kp,weights[neg_added,j].sum()/kn,atol=1e-12)
        assert seeded_pos+fraction<=0.5+1e-12 and seeded_neg+fraction<=0.5+1e-12
        ACCEPTANCE.append({"reason":r,"positive_candidates":npos,"negative_candidates":nneg,
                           "accepted_positives":int(pos_added.sum()),"accepted_negatives":int(neg_added.sum()),
                           "added_positive_weight":float(weights[pos_added,j].sum()),"added_negative_weight":float(weights[neg_added,j].sum()),
                           "added_class_fraction":fraction})
    np.testing.assert_array_equal(targets[seed_w>0],seed_y[seed_w>0])
    np.testing.assert_array_equal(weights[seed_w>0],seed_w[seed_w>0])
    return targets,weights


def run_chain(confidence,cap):
    teacher=seed_model;previous_targets=seed_y.copy()
    for round_number in range(1,ROUNDS+1):
        name=f"confidence{confidence:g}_cap{cap:g}_round{round_number}".replace(".","p")
        teacher_probability=predict_reason_models(teacher,X)
        start=len(ACCEPTANCE)
        y,w=build_self_targets(teacher_probability,confidence,cap)
        for row in ACCEPTANCE[start:]:row.update({"experiment":name,"confidence":confidence,"weight_cap":cap,"round":round_number})
        dataset=save_dataset(name,y,w,teacher_probability)
        # Distinguish original verified labels, frozen DBSCAN seeds, and new self-training targets.
        audit=pd.read_csv(OUT/"label_audits"/f"{name}.csv")
        for j,r in enumerate(REASONS):
            audit[f"label_source_{r}"]=np.where(known,"original",np.where(seed_w[:,j]>0,"dbscan_seed",np.where(w[:,j]>0,"self_training","unknown")))
        audit.to_csv(OUT/"label_audits"/f"{name}.csv",index=False)
        with threadpool_limits(limits=1):model=fit_reason_models(name,"logistic",1000)
        model["self_training"]={"confidence":confidence,"maximum_added_example_weight":cap,"rounds":round_number,
                                "maximum_added_class_fraction":MAXIMUM_ADDED_CLASS_FRACTION,"seed_dataset":"dbscan_seed"}
        probabilities=predict_reason_models(model,X_valid)
        metrics=save_predictions(name,probabilities)
        joblib.dump(model,OUT/"classifier_models"/f"{name}.joblib",compress=3);MODELS[name]=model
        extra=(w>0)&(seed_w==0)
        previous_extra=(previous_targets!=-100)&(seed_w==0)
        flips=extra&previous_extra&(y!=previous_targets)
        prediction,_=label_predictions(probabilities)
        ROWS.append({"experiment":name,"confidence":confidence,"weight_cap":cap,"round":round_number,
                     "generated_trials_total":dataset["generated_trials"],"self_labeled_trials":int(extra.any(axis=1).sum()),
                     "self_labeled_reason_slots":int(extra.sum()),"dropped_reason_slots_vs_previous":int((previous_extra&~extra).sum()),
                     "flipped_reason_slots_vs_previous":int(flips.sum()),**metrics,
                     "changed_trials_vs_seed":int((prediction!=seed_prediction).any(axis=1).sum()),
                     "delta_vs_seed":metrics["sample_f1"]-seed_metrics["sample_f1"],"kaggle_score":np.nan})
        pd.DataFrame(ROWS).to_csv(OUT/"comparison.csv",index=False)
        pd.DataFrame(ACCEPTANCE).to_csv(OUT/"acceptance_audit.csv",index=False)
        pd.DataFrame(LABEL_COUNTS).to_csv(OUT/"label_counts.csv",index=False)
        print(f"{name}: sample F1={metrics['sample_f1']:.6f}, macro F1={metrics['macro_f1']:.6f}, "
              f"new reason slots={int(extra.sum())}, change={ROWS[-1]['delta_vs_seed']:+.6f}",flush=True)
        teacher=model;previous_targets=y


In [4]:
for cap in SELF_WEIGHT_CAPS:run_chain(0.90,cap)


confidence0p9_cap0p05_round1: sample F1=0.897436, macro F1=0.805022, new reason slots=4717, change=-0.012821


confidence0p9_cap0p05_round2: sample F1=0.897436, macro F1=0.801106, new reason slots=4797, change=-0.012821


confidence0p9_cap0p1_round1: sample F1=0.897436, macro F1=0.805022, new reason slots=4717, change=-0.012821


confidence0p9_cap0p1_round2: sample F1=0.897436, macro F1=0.801106, new reason slots=4802, change=-0.012821


In [5]:
for cap in SELF_WEIGHT_CAPS:run_chain(0.95,cap)


confidence0p95_cap0p05_round1: sample F1=0.897436, macro F1=0.805022, new reason slots=4649, change=-0.012821


confidence0p95_cap0p05_round2: sample F1=0.897436, macro F1=0.805022, new reason slots=4723, change=-0.012821


confidence0p95_cap0p1_round1: sample F1=0.897436, macro F1=0.805022, new reason slots=4649, change=-0.012821


confidence0p95_cap0p1_round2: sample F1=0.901709, macro F1=0.812040, new reason slots=4734, change=-0.008547


In [6]:
comparison=pd.DataFrame(ROWS);assert len(comparison)==8
ranked=comparison.sort_values(["sample_f1","round","weight_cap","confidence"],ascending=[False,True,True,False],kind="stable")
best=ranked.iloc[0]
selected=joblib.load(OUT/"classifier_models"/f"{best.experiment}.joblib")
selected_probability=predict_reason_models(selected,X_valid)
for row in comparison.itertuples(index=False):
    frame=pd.read_csv(OUT/"datasets"/f"{row.experiment}.csv",float_precision="round_trip")
    y=frame[REASONS].to_numpy(dtype=int);w=frame[[f"weight_{r}" for r in REASONS]].to_numpy()
    assert frame.ID.tolist()==train.ID.tolist()
    np.testing.assert_array_equal(y[seed_w>0],seed_y[seed_w>0]);np.testing.assert_array_equal(w[seed_w>0],seed_w[seed_w>0])
    prediction_file=pd.read_csv(OUT/"validation_predictions"/f"{row.experiment}.csv",float_precision="round_trip")
    assert prediction_file.ID.tolist()==valid.ID.tolist()
    probs=prediction_file[[f"probability_{r}" for r in REASONS]].to_numpy()
    predicted=prediction_file[[f"predicted_{r}" for r in REASONS]].to_numpy(dtype=int)
    np.testing.assert_allclose(predict_reason_models(joblib.load(OUT/"classifier_models"/f"{row.experiment}.joblib"),X_valid),probs,rtol=1e-12,atol=1e-12)
    np.testing.assert_array_equal(label_predictions(probs)[0],predicted)
    np.testing.assert_allclose(f1_score(truth,predicted,average="samples",zero_division=0),row.sample_f1,atol=1e-14)
    np.testing.assert_allclose(f1_score(truth,predicted,average="macro",zero_division=0),row.macro_f1,atol=1e-14)
frame=pd.read_csv(OUT/"datasets"/f"{best.experiment}.csv",float_precision="round_trip")
scaled=classifier_scaler.transform(frame[FEATURES].to_numpy());replay={**selected,"estimators":[]}
with threadpool_limits(limits=1):
    for j,r in enumerate(REASONS):
        y=frame[r].to_numpy(dtype=int);w=frame[f"weight_{r}"].to_numpy();included=w>0
        estimator=clone(selected["estimators"][j])
        with warnings.catch_warnings():
            warnings.simplefilter("error",ConvergenceWarning)
            estimator.fit(scaled[included],y[included],sample_weight=w[included]*157/w[included].sum())
        replay["estimators"].append(estimator)
np.testing.assert_allclose(predict_reason_models(replay,X_valid),selected_probability,rtol=1e-8,atol=1e-10)
np.testing.assert_array_equal(label_predictions(predict_reason_models(replay,X_valid))[0],label_predictions(selected_probability)[0])
for rel,h in protected_hashes.items():assert hashlib.sha256((DATA/rel).read_bytes()).hexdigest()==h
print("All seed-preservation, saved-model, score, independent-refit, and protected-input checks passed.")
display(comparison[["confidence","weight_cap","round","self_labeled_trials","self_labeled_reason_slots","sample_f1","macro_f1","delta_vs_seed"]].round(6))


All seed-preservation, saved-model, score, independent-refit, and protected-input checks passed.


,confidence,weight_cap,round,self_labeled_trials,self_labeled_reason_slots,sample_f1,macro_f1,delta_vs_seed
0,0.90,0.05,1,1014,4717,0.897436,0.805022,-0.012821
1,0.90,0.05,2,1014,4797,0.897436,0.801106,-0.012821
2,0.90,0.10,1,1014,4717,0.897436,0.805022,-0.012821
3,0.90,0.10,2,1014,4802,0.897436,0.801106,-0.012821
4,0.95,0.05,1,1014,4649,0.897436,0.805022,-0.012821
5,0.95,0.05,2,1014,4723,0.897436,0.805022,-0.012821
6,0.95,0.10,1,1014,4649,0.897436,0.805022,-0.012821
7,0.95,0.10,2,1014,4734,0.901709,0.812040,-0.008547


In [7]:
candidate=None
if best.sample_f1>seed_metrics["sample_f1"]+1e-12:
    test=pd.read_csv(INPUT_DIR/"part1_test.csv");sample=pd.read_csv(INPUT_DIR/"part1_sample_submission.csv")
    assert test.ID.is_unique and sample.ID.is_unique and set(test.ID)==set(sample.ID)
    assert set(test.ID).isdisjoint(train.ID) and set(test.ID).isdisjoint(valid.ID)
    prediction,empty=label_predictions(predict_reason_models(selected,test[FEATURES].to_numpy()))
    output=pd.DataFrame({"ID":test.ID,"Predicted":[" ".join(r for r,yes in zip(REASONS,row) if yes) for row in prediction]})
    output=output.set_index("ID").loc[sample.ID].reset_index()
    assert output.shape==(312,2) and output.Predicted.str.len().gt(0).all()
    path=OUT/"submissions"/f"part1_self_training_{best.experiment}_submission.csv"
    output.to_csv(path,index=False);pd.testing.assert_frame_equal(pd.read_csv(path),output)
    candidate={"experiment":best.experiment,"validation_sample_f1":float(best.sample_f1),"validation_macro_f1":float(best.macro_f1),
               "submission_file":str(path.relative_to(OUT)),"submission_sha256":hashlib.sha256(path.read_bytes()).hexdigest(),
               "model_file":f"classifier_models/{best.experiment}.joblib","model_sha256":hashlib.sha256((OUT/"classifier_models"/f"{best.experiment}.joblib").read_bytes()).hexdigest(),
               "test_rows":312,"test_fallback_trials":int(empty.sum()),"kaggle_score":None,"kaggle_status":"prepared locally; not submitted"}
    (OUT/"submission_manifest.json").write_text(json.dumps(candidate,indent=2)+"\n")
settings={"confidences":CONFIDENCES,"additional_weight_caps":SELF_WEIGHT_CAPS,"rounds":ROUNDS,"configurations":8,
          "maximum_added_class_fraction":MAXIMUM_ADDED_CLASS_FRACTION,"maximum_total_pseudo_class_fraction":0.5,
          "seed_targets_preserved":True,"seed_weights_preserved":True,"refresh_from_original_seed_unknown_slots_each_round":True,
          "verified_training_labels":157,"validation_labels_added_to_training":False,"validation_used_for_fit":False,"test_used_for_fit":False,
          "features":FEATURES,"classifier_C":1000,"classifier_scaler_fit_rows":157,"fitting_weight_total_per_reason":157,
          "prediction_cutoff":0.5,"fallback":True,"reference_sample_f1":seed_metrics["sample_f1"],"reference_macro_f1":seed_metrics["macro_f1"],
          "best_self_training_experiment":best.experiment,"best_self_training_sample_f1":float(best.sample_f1),"best_self_training_macro_f1":float(best.macro_f1),
          "leader_changed":bool(best.sample_f1>seed_metrics["sample_f1"]+1e-12),"candidate":candidate,
          "protected_input_sha256":protected_hashes,"dataset_file_sha256":{str(p.relative_to(OUT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted((OUT/"datasets").glob('*.csv'))},
          "versions":{"python":platform.python_version(),"sklearn":sklearn.__version__,"numpy":np.__version__,"pandas":pd.__version__},"new_kaggle_scores":None}
(OUT/"settings.json").write_text(json.dumps(settings,indent=2)+"\n")
lines=["# Conservative self-training from the DBSCAN seed","",
       "Eight settings cross confidence 0.90/0.95, added-example caps 0.05/0.10, and one/two rounds. "
       "Only original seed-unknown reason slots in unlabeled training rows can receive new targets. Known and DBSCAN seed "
       "targets/weights remain unchanged. Both new pseudo classes are required; their total added fractions preserve the "
       "known class ratio, capped at 0.25 per class and total seed+self influence 0.50. Each round refreshes its additions "
       "from the original seed using the previous classifier, allowing uncertain self-generated targets to be dropped.","",
       f"Seed sample F1 **{seed_metrics['sample_f1']:.6f}**, macro F1 **{seed_metrics['macro_f1']:.6f}**. "
       f"Best self-training setting **{best.experiment}**: sample F1 **{best.sample_f1:.6f}**, macro F1 **{best.macro_f1:.6f}**.","",
       "| Confidence | Added weight cap | Round | Self-labeled trials | Self-labeled reason slots | Sample F1 | Macro F1 | Change |",
       "|---:|---:|---:|---:|---:|---:|---:|---:|"]
for row in comparison.itertuples(index=False):
    lines.append(f"| {row.confidence:g} | {row.weight_cap:g} | {row.round} | {row.self_labeled_trials} | {row.self_labeled_reason_slots} | {row.sample_f1:.6f} | {row.macro_f1:.6f} | {row.delta_vs_seed:+.6f} |")
lines.extend(["","Classifier/scaler, 150 embeddings, C=1000, fitting-weight total 157, cutoff 0.5, and fallback stay fixed. "
       "Validation and test do not participate in training. No final refit with validation labels was performed. "
       "These are exploratory selections on the same 78 validation trials; no independent test improvement is established.","",
       "Exact datasets/weights, source audits distinguishing original/DBSCAN/self-training targets, teacher probabilities, "
       "classifiers, validation predictions, per-reason acceptance counts, settings, and checksums are saved. "
       "The seed reproduces the current leader. Every saved model rescored correctly and preserves all seed targets/weights; "
       "the selected model independently refits from its saved data with matching probabilities. Protected sources retain their hashes.","",
       "No Kaggle submission, commit, or tag was made.",""])
lines.append(f"New candidate prepared locally: `{candidate['submission_file']}`. Kaggle score unknown." if candidate else
             "No self-training setting improves the current leader; keep the seed model.")
(OUT/"SUMMARY.md").write_text("\n".join(lines)+"\n")
print("Best self-training:",best.experiment,"score:",best.sample_f1,"Leader changed:",settings['leader_changed'])


Best self-training: confidence0p95_cap0p1_round2 score: 0.9017094017094018 Leader changed: False
